In [1]:
"""
Load, plot, and sniff through SolO SWE instrument data. 
Requires following packages: pyspedas, astropy, scipy, numpy, panda if not already installed.
Initial code written by Hadi, 2026-05-04
"""

import sys 
import pyspedas
from pyspedas import tplot, tplot_options, del_data, get_data, store_data
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib as mpl
import matplotlib.pyplot as plt
import itertools
from scipy.ndimage import median_filter
from scipy.signal import find_peaks
import matplotlib.dates as mdates
from datetime import datetime, timezone
%matplotlib inline


In [ ]:
# trr=['2021-10-01', '2021-10-29']
trr=['2022-09-03', '2022-09-15']
pyspedas.timespan(trr[0], 5, units='days', reset=True)
mag_vars = pyspedas.projects.solo.mag(trange=trr, datatype='rtn-normal')
swa_vars = pyspedas.projects.solo.swa(trange=trr)
swa_vars = pyspedas.projects.solo.swa(trange=trr, datatype='pas-grnd-mom')
# pyspedas.tplot_names()


In [8]:
B_rtn_SolO=get_data('B_RTN')
bmags=np.linalg.norm(B_rtn_SolO[1], axis=1)
store_data("B_RTN_mag",{'x':B_rtn_SolO[0], 'y':bmags})
store_data("B_RTN_All", data=["B_RTN", "B_RTN_mag"])
pyspedas.options('B_RTN_All', 'yrange', [-50,50])
pyspedas.options('B_RTN_All', 'legend_names', ['Br','Bt','Bn','|B|'])

#### Plot variables
pyspedas.tlimit(['2022-09-05', '2022-09-15'])
pyspedas.tplot_options('data_gap', 60)
myfig, ax = tplot(['B_RTN_All', 'eflux', 'N'], return_plot_objects=True)

In [19]:

# for this finder algorithm to work, you essentially need |B|, N, |V|, T data.
# Each variable structure should have a "times" and "y" strucutre elements.
# It then goes through the time series and finds instances of simultaneous jumps
# in at least 3 of those variables. |B| and N should always 

def _sliding_ratio(signal, half_win):
    """
    This is how ratios are calculated at each data points. 
    (Data from B, N, V, T are interped to a common time array)
    upstream[i]   = mean(signal[i-half_win : i])   — trailing window, shifted
    downstream[i] = mean(signal[i : i+half_win])   — leading window via reversal
    """
    s    = pd.Series(signal.astype(float))
    up   = s.rolling(half_win, min_periods=1).mean().shift(1)
    down = pd.Series(
        s[::-1].rolling(half_win, min_periods=1).mean().values[::-1], index=s.index
    )
    with np.errstate(invalid='ignore', divide='ignore'):
        ratio = down / up
    ratio[up <= 0] = np.nan
    return ratio.values


def _adaptive_sig(ratio, half_win, n_sigma):
    """
    Boolean mask: True where ratio exceeds local_median + n_sigma × local_MAD.
    Background estimated over a window 12× wider than the shock window.
    """
    bg     = max(3, half_win * 12 + 1)
    filled = np.where(np.isnan(ratio), np.nanmedian(ratio), ratio)
    med    = median_filter(filled, size=bg, mode='reflect')
    mad    = median_filter(np.abs(filled - med), size=bg, mode='reflect')
    return np.abs(ratio - med) > n_sigma * np.maximum(mad, 1e-10)


def _single_pass(B, N, V, T, t, half_win, min_sep, required_vars, min_total, n_sigma):
    """
    One detection pass for a given (window, n_sigma) combination.
    Returns a list of candidate dicts.
    """
    ratios = {k: _sliding_ratio(v, half_win) for k, v in zip('BNVT', [B, N, V, T])}
    sig    = {k: _adaptive_sig(r, half_win, n_sigma) for k, r in ratios.items()}

    req_mask  = np.all([sig[v] for v in required_vars], axis=0)
    n_sig     = np.sum([sig[k].astype(int) for k in 'BNVT'], axis=0)
    all_valid = ~np.any([np.isnan(r) for r in ratios.values()], axis=0)

    # score = np.where(all_valid, np.sum(list(ratios.values()), axis=0), 0.0)
    score = np.where(all_valid, np.sum([np.abs(r - 1) for r in ratios.values()], axis=0), 0.0)
    score = np.where(req_mask & (n_sig >= min_total), score, 0.0)

    peaks, _ = find_peaks(score, distance=min_sep)

    return [
        {'time': t[i], **{f'r_{k}': float(ratios[k][i]) for k in 'BNVT'},
         'n_sig': int(n_sig[i])}
        for i in peaks if score[i] > 0
    ]


# =============================================================================
# The following loops through both sliding window and jump thresholds (e.g., sigmas)
# =============================================================================

def detect_shocks_auto(
    window_minutes_list = [5, 10, 20, 30],
    n_sigma_list        = [3, 4, 5],
    min_sep_minutes     = 30,
    required_vars       = ['B', 'N'],
    min_total           = 3,
    min_votes           = 2,
):
    """
    Detect interplanetary shocks by assessing jump (window_size × n_sigma) combinations.
    
    Parameters
    ----------
    window_minutes_list : list of float   — window half-widths to probe [minutes]
    n_sigma_list        : list of float   — adaptive threshold sensitivities to sweep
    min_sep_minutes     : float           — minimum separation between events [minutes]
    required_vars       : list of str     — variables in ['B','N','V','T'] that must jump
    min_total           : int             — minimum total variables that must jump (of 4)
    min_votes           : int             — minimum combinations that must agree

    For each pair (window, n_sigma), the detector runs independently and casts
    a vote for any shock candidate it finds.  A candidate is confirmed only if
    it collects >= min_votes across all combinations.

    With default settings (4 windows × 3 sigma levels = 12 combinations):
      - A strong, sharp shock scores up to 12 votes.
      - A real but weak shock may score 3-6.
      - Noise typically scores 1.

    Looping through 4 different windows and 3 sigma levels seems to flag most shocks, plus
    it minimizes the number of false flags. These numbers can be changed.
    
    If no events pass, then it automatically reduces min_votes, then min_total.

    """
    # ── Load and prepare data ────────────────────────────────────────────────
    B_d, N_d, V_d, T_d = get_data('B_RTN'), get_data('N'), get_data('V_RTN'), get_data('T')
    if any(d is None for d in [B_d, N_d, V_d, T_d]):
        raise RuntimeError("Missing tplot variables either B_RTN, N, V_RTN, T are not loaded.")

    t = N_d.times   # common grid: plasma (SWA) cadence
    B = np.interp(t, B_d.times, np.linalg.norm(B_d.y, axis=1))
    N = N_d.y.squeeze()
    V = np.interp(t, V_d.times, np.linalg.norm(V_d.y, axis=1))
    T = np.interp(t, T_d.times, T_d.y.squeeze())

    dt      = np.nanmedian(np.diff(t))
    min_sep = max(1, int(round(min_sep_minutes * 60 / dt)))
    n_combos = len(window_minutes_list) * len(n_sigma_list)

    fallbacks = list(dict.fromkeys([           # remove duplicates, keep order
        (min_votes,           min_total),
        (max(1, min_votes-1), min_total),
        (1,                   min_total),
        (1,                   max(2, min_total - 1)),
        (1,                   2),
    ]))

    for attempt, (votes_req, mtotal) in enumerate(fallbacks):

    # ── Collect candidates from all (window, n_sigma) combinations
        all_cands = []
        for wm, ns in itertools.product(window_minutes_list, n_sigma_list):
            hw = max(1, int(round(wm * 60 / dt)))
            all_cands.extend(_single_pass(B, N, V, T, t, hw, min_sep,
                                          required_vars, mtotal, ns))

        if not all_cands:
            continue

    # ── Grouping candidates within min_sep_minutes and count votes
        cands_sorted = sorted(all_cands, key=lambda c: c['time'])
        clusters, group = [], [cands_sorted[0]]
        for c in cands_sorted[1:]:
            if c['time'] - group[0]['time'] <= min_sep_minutes * 60:
                group.append(c)
            else:
                clusters.append(group); group = [c]
        clusters.append(group)

        rows = []
        for group in clusters:
            if len(group) < votes_req:
                continue
            # Calculating best, medium and worst (low) candidates
            best  = max(group, key=lambda c: sum(c[f'r_{k}'] for k in 'BNVT'))
            votes = len(group)
            conf  = ('High'   if votes >= n_combos * 0.6 else
                     'Medium' if votes >= n_combos * 0.25 else 'Low')
            rows.append({
                'time_unix'  : best['time'],
                'time_str'   : pd.Timestamp(best['time'], unit='s', tz='UTC')
                               .strftime('%Y-%m-%d %H:%M:%S UTC'),
                'shock_type' : 'Forward' if best['r_N'] >= 1.0 else 'Reverse',
                'ratio_B'    : round(best['r_B'], 3),
                'ratio_N'    : round(best['r_N'], 3),
                'ratio_V'    : round(best['r_V'], 3),
                'ratio_T'    : round(best['r_T'], 3),
                'n_vars'     : best['n_sig'],
                'votes'      : votes,
                'confidence' : conf,
            })

        if rows:
            if attempt > 0:
                print(f"[Auto-fallback] Relaxed to min_votes={votes_req}, min_total={mtotal}.")
            return pd.DataFrame(rows)

    print("[detect_shocks_auto] No shock candidates found in the loaded time range.")
    return pd.DataFrame()



shocks = detect_shocks_auto(
    window_minutes_list = [5, 10, 20, 30],  # window sizes to probe ; This is the time window around the shock instance
    n_sigma_list        = [3, 4, 5],        # adaptive threshold sensitivities  ;  This tells how significant the jump is. Is not the jump ratio
    min_sep_minutes     = 30,
    required_vars       = ['B', 'N'],       # must always jump
    min_total           = 3,               # + at least one of V or T
    min_votes           = 2,               # must appear in >= 2 of 12 combinations
)

# Printing what is found:

if shocks is None or len(shocks) == 0:
    print("No shocks detected.")
    sys.exit()
n_combos = len([5,10,20,30]) * len([3,4,5])   # informational
print(f"\n{'='*84}")
print(f"  SHOCK CATALOG  —  {len(shocks)} event(s)   "
      f"[max possible votes: {n_combos}]")
print(f"{'='*84}")
print(f"  {'#':>3}  {'Time (UTC)':^25}  {'Type':^9}  "
      f"{'|B|':^6}  {'N':^6}  {'|V|':^6}  {'T':^6}  {'Votes':^5}  {'Conf':^6}")
print(f"  {'-'*79}")
for i, row in shocks.iterrows():
    print(f"  {i+1:>3}  {row['time_str']:^25}  {row['shock_type']:^9}  "
          f"{row['ratio_B']:^6.2f}  {row['ratio_N']:^6.2f}  "
          f"{row['ratio_V']:^6.2f}  {row['ratio_T']:^6.2f}  "
          f"{row['votes']:^5}  {row['confidence']:^6}")
print(f"{'='*84}")








  SHOCK CATALOG  —  7 event(s)   [max possible votes: 12]
    #         Time (UTC)            Type      |B|      N      |V|      T     Votes   Conf 
  -------------------------------------------------------------------------------
    1   2022-09-03 03:56:21 UTC    Forward    1.11    1.33    1.01    1.21     2     Low  
    2   2022-09-04 11:51:25 UTC    Forward    1.10    1.13    0.98    1.69     2     Low  
    3   2022-09-04 14:03:21 UTC    Forward    1.31    1.49    1.09    1.81     2     Low  
    4   2022-09-06 10:00:33 UTC    Forward    1.77    1.67    1.67    5.07     5    Medium
    5   2022-09-08 04:09:53 UTC    Forward    1.31    1.84    1.03    2.24    11     High 
    6   2022-09-08 21:00:45 UTC    Forward    1.59    1.93    1.20    3.08    12     High 
    7   2022-09-13 17:10:38 UTC    Forward    1.42    1.38    1.05    1.32    11     High 
